# Pathway Enrichment Analysis with Reactome

This notebook demonstrates how to compute pathway enrichment scores for drug treatments using GSEApy prerank with Reactome pathways.

In [ ]:
from l2l_bench import L2LData, PathwayEnrichment

data = L2LData()
enrichment = PathwayEnrichment(data)

## Compute Enrichment for One Treatment

Let's compute pathway enrichment for Trametinib (a MEK inhibitor) in A549 cells.

In [ ]:
results = enrichment.compute_enrichment(
    drug="Trametinib",
    concentration=0.05,
    cell_line="A549"
)
print(f"Computed {len(results)} pathway scores")
results.head(10)

## Get Specific Pathway Score

We can retrieve the score for a specific pathway using partial name matching.

In [ ]:
mapk_score = enrichment.get_pathway_score(
    pathway="MAPK",
    drug="Trametinib",
    concentration=0.05,
    cell_line="A549"
)
print(f"MAPK pathway NES: {mapk_score['nes']:.3f} (FDR: {mapk_score['fdr']:.3e})")

## List Available Pathways

In [ ]:
pathways = enrichment.list_pathways("Trametinib", 0.05, "A549")
print(f"Available pathways: {len(pathways)}")
print("\nFirst 10 pathways:")
for p in pathways[:10]:
    print(f"  - {p}")

## Filter for Significant Pathways

Let's find the most significantly enriched pathways (both up and down regulated).

In [ ]:
sig_pathways = results[results['fdr'] < 0.25].sort_values('nes')

print("Top upregulated pathways:")
print(sig_pathways[sig_pathways['nes'] > 0].tail(5)[['pathway', 'nes', 'fdr']])

print("\nTop downregulated pathways:")
print(sig_pathways[sig_pathways['nes'] < 0].head(5)[['pathway', 'nes', 'fdr']])

## Biological Validation

Trametinib is a MEK inhibitor, so we expect to see:
- MAPK pathway downregulated (negative NES)
- Cell cycle pathways potentially affected

Let's check if our results are consistent with this expectation.

In [ ]:
# check MAPK-related pathways
mapk_pathways = results[results['pathway'].str.contains('MAPK|ERK|MEK', case=False)]
print("MAPK-related pathways:")
print(mapk_pathways[['pathway', 'nes', 'fdr']].sort_values('nes'))